In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("priority-scoring")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/24 18:36:16 WARN Utils: Your hostname, CrisBook.local, resolves to a loopback address: 127.0.0.1; using 192.168.13.159 instead (on interface en0)
26/04/24 18:36:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/24 18:36:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
master_df = spark.read.parquet("../data/dev/gold/master_vulnerabilities")

master_df.printSchema()
master_df.show(5, truncate=False)

root
 |-- cve_id: string (nullable = true)
 |-- description: string (nullable = true)
 |-- cwe: string (nullable = true)
 |-- published: string (nullable = true)
 |-- lastModified: string (nullable = true)
 |-- cvss_score: double (nullable = true)
 |-- cvss_severity: string (nullable = true)
 |-- kev_date_added: date (nullable = true)
 |-- required_action: string (nullable = true)
 |-- known_ransomware_campaign_use: string (nullable = true)
 |-- epss_score: double (nullable = true)
 |-- epss_percentile: double (nullable = true)
 |-- is_kev: integer (nullable = true)

+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-----------------------+-----------------------+----------

In [3]:
from pyspark.sql.functions import col, coalesce, lit
from pyspark.sql.types import DoubleType, IntegerType

scoring_df = (
    master_df
    .withColumn("cvss_score", coalesce(col("cvss_score").cast(DoubleType()), lit(0.0)))
    .withColumn("epss_score", coalesce(col("epss_score").cast(DoubleType()), lit(0.0)))
    .withColumn("is_kev", coalesce(col("is_kev").cast(IntegerType()), lit(0)))
)

In [4]:
scoring_df = scoring_df.withColumn(
    "cvss_normalized",
    col("cvss_score") / 10
)

In [5]:
from pyspark.sql.functions import round

scoring_df = scoring_df.withColumn(
    "priority_score",
    round(
        (lit(0.4) * col("cvss_normalized")) +
        (lit(0.4) * col("epss_score")) +
        (lit(0.2) * col("is_kev")),
        4
    )
)

In [6]:
scoring_df.orderBy(col("priority_score").desc()).select(
    "cve_id",
    "cvss_score",
    "cvss_severity",
    "epss_score",
    "epss_percentile",
    "is_kev",
    "priority_score",
    "description"
).show(20, truncate=False)

+--------------+----------+-------------+----------+---------------+------+--------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
from pyspark.sql.functions import when

scoring_df = scoring_df.withColumn(
    "priority_level",
    when(col("priority_score") >= 0.8, "Critical")
    .when(col("priority_score") >= 0.6, "High")
    .when(col("priority_score") >= 0.4, "Medium")
    .otherwise("Low")
)

In [8]:
scoring_df.groupBy("priority_level").count().orderBy("priority_level").show()

+--------------+-----+
|priority_level|count|
+--------------+-----+
|      Critical|  135|
|          High|  517|
|           Low|93479|
|        Medium| 1848|
+--------------+-----+



In [9]:
baseline_df = scoring_df.withColumn(
    "cvss_only_score",
    round(col("cvss_normalized"), 4)
)

In [10]:
baseline_df.orderBy(col("cvss_only_score").desc()).select(
    "cve_id",
    "cvss_score",
    "epss_score",
    "is_kev",
    "priority_score",
    "cvss_only_score"
).show(20, truncate=False)

+--------------+----------+----------+------+--------------+---------------+
|cve_id        |cvss_score|epss_score|is_kev|priority_score|cvss_only_score|
+--------------+----------+----------+------+--------------+---------------+
|CVE-2026-1633 |10.0      |9.3E-4    |0     |0.4004        |1.0            |
|CVE-2025-27364|10.0      |0.21011   |0     |0.484         |1.0            |
|CVE-2024-25913|10.0      |0.00771   |0     |0.4031        |1.0            |
|CVE-2025-30012|10.0      |0.01772   |0     |0.4071        |1.0            |
|CVE-2026-28409|10.0      |0.00749   |0     |0.403         |1.0            |
|CVE-2025-24201|10.0      |0.00225   |1     |0.6009        |1.0            |
|CVE-2024-22004|10.0      |7.0E-4    |0     |0.4003        |1.0            |
|CVE-2025-22612|10.0      |0.0052    |0     |0.4021        |1.0            |
|CVE-2026-25725|10.0      |2.3E-4    |0     |0.4001        |1.0            |
|CVE-2025-26701|10.0      |0.0018    |0     |0.4007        |1.0            |

In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, sum as spark_sum

w_priority = Window.orderBy(col("priority_score").desc())

ranked_df = scoring_df.withColumn(
    "priority_rank",
    row_number().over(w_priority)
)

In [12]:
for k in [100, 500, 1000]:
    kev_in_top_k = (
        ranked_df
        .filter(col("priority_rank") <= k)
        .agg(spark_sum("is_kev").alias("kev_count"))
        .collect()[0]["kev_count"]
    )

    print(f"KEV vulnerabilities in top {k}: {kev_in_top_k}")

26/04/24 18:38:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 18:38:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


KEV vulnerabilities in top 100: 100
KEV vulnerabilities in top 500: 200
KEV vulnerabilities in top 1000: 328


26/04/24 18:38:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 18:38:40 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 18:38:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/04/24 18:38:41 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [13]:
scoring_df.write.mode("overwrite").parquet("../data/dev/gold/vulnerability_scores")

In [15]:
scores_check = spark.read.parquet("../data/dev/gold/vulnerability_scores")
scores_check.show(5, truncate=False)

+--------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+-----------------------+-----------------------+----------+-------------+--------------+---------------+-----------------------------+----------+---------------+------+---------------+--------------+--------------+
|cve_id        |description                                                                                                                                                                                                                                                                                                                                             |cwe   |published              |lastModified           |c

26/04/25 00:15:19 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 934820 ms exceeds timeout 120000 ms
26/04/25 00:15:19 WARN SparkContext: Killing executors is not supported by current scheduler.
26/04/25 00:15:19 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at o